# Choosing a deployable pseudo-label gate

Companion to [`glem_results.ipynb`](glem_results.ipynb) and the preregistration in
[`EXPERIMENT.md`](../EXPERIMENT.md).

The oracle sweep established that perfect per-node gating is worth something — pubmed
+0.9pp (seed-stable), cora +3.9pp over a size-matched random control. The oracle is
unreachable, because it *selects* using gold labels. This notebook asks what a
**deployable** gate could achieve instead, and which signal it should use.

Everything here runs off the existing archive in `temp/probe_output/`. No training.

## Settling the gate offline — which signal should a real gate use?

The oracle sweep showed perfect gating is worth something (pubmed +0.9pp seed-stable,
cora +3.9pp). The oracle is unreachable, though, because it *selects* using gold labels.
This section asks what a **deployable** gate could achieve, entirely from the existing
archive — no training.

**What counts as deployable.** The gate must score a node without knowing its label.
That admits the teacher's own distribution (confidence, entropy, margin), GLANCE soft
homophily, kNN ambiguity (neighbours drawn from *train* labels, which are known), degree,
and teacher–student agreement. It **excludes true local homophily** — that needs the
node's own label, so the 0.845 AUROC it scores is not attainable in deployment.

**Fit on val, evaluate on test.** A learned gate needs labelled nodes to fit on. Train is
unusable: the GNN memorises its train labels, so its error profile there is fiction.
Teacher accuracy by split makes this stark —

| dataset | direction | train | val | test |
|---|---|---|---|---|
| citeseer | gnn→lm | **1.000** | 0.554 | 0.496 |
| cornell | gnn→lm | **1.000** | 0.754 | 0.789 |
| cora | gnn→lm | 0.993 | 0.908 | 0.884 |
| arxiv | gnn→lm | 0.831 | 0.768 | 0.766 |

Val tracks test closely; train does not. Using val labels is legitimate — GLEM already
uses them for early stopping — so the gate is fit on val and scored on test throughout.

### Features, including two built from the pseudo-labels

The preregistered `ambiguity` signal draws its kNN neighbours only from **train** nodes —
4% of the graph on citeseer. Pseudo-labels lift that restriction, since every node then
carries a label, and they stay deployable because they are just model predictions. Two
were added:

- **`nbr_pl_agree`** — fraction of graph neighbours carrying the same pseudo-label. A
  hard-label counterpart to GLANCE, which uses the soft distributions and can be
  dominated by a confident-but-wrong peak.
- **`knn_pl_entropy`** — kNN pseudo-label entropy over **all** nodes rather than the
  labelled train set.

`knn_pl_entropy` is O(N²) in the neighbour search, so it is skipped above
`KNN_PL_MAX_N` (arxiv), where it would need a GPU. The column reads NaN there rather
than silently switching to a sampled neighbour pool that would not be comparable.

In [ ]:
"""Offline gate selection: which DEPLOYABLE score best predicts 'teacher is wrong'?

Deployable means computable without the node's own label. That admits the teacher's
own distribution (confidence / entropy / margin), GLANCE soft homophily (predicted
distributions only), kNN ambiguity (drawn from TRAIN labels, which are known),
degree, and teacher-student agreement. It excludes true local homophily, which
needs the node's label -- the 0.845 AUROC quoted earlier came from that and is not
attainable in deployment.

Fit on VAL, evaluate on TEST. Not on train: the GNN memorises its train labels, so
its error profile there is unrepresentative (citeseer 1.000 train vs 0.496 test),
and a gate fit on it would be calibrated to a regime it never sees. Val is
legitimate -- GLEM already uses those labels for early stopping.
"""
import json
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path.cwd()
while not (ROOT / 'src' / 'probe').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
import sys; sys.path.insert(0, str(ROOT / 'src'))
PROBE = ROOT / 'temp' / 'probe_output'; SIG = PROBE / '_signals'
from probe.signals import soft_local_homophily, load_tag_graph


def fast_auroc(scores, labels):
    labels = np.asarray(labels).astype(bool)
    npos, nneg = int(labels.sum()), int((~labels).sum())
    if npos == 0 or nneg == 0:
        return float('nan')
    order = np.argsort(scores, kind='mergesort')
    ranks = np.empty(len(scores)); ranks[order] = np.arange(1, len(scores) + 1)
    s = scores[order]; i = 0
    while i < len(s):
        j = i + 1
        while j < len(s) and s[j] == s[i]:
            j += 1
        if j - i > 1:
            ranks[order[i:j]] = (i + 1 + j) / 2.0
        i = j
    return float((ranks[labels].sum() - npos * (npos + 1) / 2.0) / (npos * nneg))


def edge_index_for(key):
    f = SIG / f'{key}_edge_index.npy'
    if f.exists():
        return np.load(f)
    ei, _, _ = load_tag_graph(key)
    ei = np.asarray(ei); np.save(f, ei)
    return ei


def features_for_step(run, st, key):
    """Deployable per-node features + the target (teacher is wrong)."""
    z = np.load(run / 'splits.npz'); y = z['labels']
    T = np.load(run / 'teacher' / f"step{st['step_index']}_{st['em_phase']}.npy").astype(np.float32)
    p = np.exp(T - T.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
    srt = np.sort(p, axis=1)
    C = p.shape[1]

    kind = 'lm' if st['student'] == 'LM' else 'gnn'
    bf = run / 'logits' / f"iter{st['em_iter'] - 1}_{kind}.npy"
    student = np.load(bf).astype(np.float32).argmax(1) if bf.exists() else p.argmax(1)

    ei = edge_index_for(key)
    deg = np.bincount(ei[0], minlength=len(y)).astype(np.float32)
    amb = np.load(SIG / f'{key}_standard_s0_ambiguity.npy')
    glance = soft_local_homophily(ei, p)

    # --- features built from the PSEUDO-LABELS themselves ---
    # Both are deployable: they use only model predictions, the graph, and text
    # embeddings. Neither needs the node's own label.
    pl = p.argmax(1)
    # (a) hard neighbourhood agreement: fraction of graph neighbours carrying the
    # same pseudo-label. A hard-label counterpart to GLANCE, which uses the soft
    # distributions and can be dominated by a confident-but-wrong peak.
    src, dst = ei[0], ei[1]
    same = (pl[src] == pl[dst]).astype(np.float64)
    num = np.zeros(len(y)); den = np.zeros(len(y))
    np.add.at(num, src, same); np.add.at(den, src, 1.0)
    nbr_pl_agree = np.divide(num, den, out=np.full(len(y), np.nan), where=den > 0)
    nbr_pl_agree = np.nan_to_num(nbr_pl_agree, nan=np.nanmedian(nbr_pl_agree))
    # (b) kNN pseudo-label entropy over ALL nodes, not just the labelled train set.
    # The preregistered `ambiguity` draws neighbours from train only -- 4% of the
    # graph on citeseer -- so this is the same idea on a far denser neighbour pool.
    emb = np.load(SIG / f'{key}_sbert.npy').astype(np.float32)
    emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-12)
    knn_pl_ent = np.empty(len(y))
    k = 15
    for a in range(0, len(emb), 2048):
        b = min(a + 2048, len(emb))
        sims = emb[a:b] @ emb.T
        top = np.argpartition(-sims, kth=k, axis=1)[:, :k + 1]
        for r_, gi in enumerate(range(a, b)):
            cand = top[r_][top[r_] != gi][:k]
            q = np.bincount(pl[cand], minlength=C).astype(np.float64)
            q /= q.sum()
            knn_pl_ent[gi] = -(q * np.log(q + 1e-12)).sum() / np.log(C)

    F = {
        'nbr_pl_agree': nbr_pl_agree,
        'knn_pl_entropy': knn_pl_ent,
        'conf': p.max(1),
        'entropy': -(p * np.log(p + 1e-12)).sum(1) / np.log(C),
        'margin': srt[:, -1] - srt[:, -2],
        'glance': np.nan_to_num(glance, nan=np.nanmedian(glance)),
        'ambiguity': np.nan_to_num(amb, nan=np.nanmedian(amb)),
        'log_degree': np.log1p(deg),
        'agrees_with_student': (p.argmax(1) == student).astype(np.float32),
    }
    return F, (p.argmax(1) != y), z


def evaluate_gates(datasets, seeds=(0,)):
    """AUROC on TEST for each candidate, fit on VAL where fitting is needed."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    SINGLE = {'-conf': ('conf', -1), '-glance': ('glance', -1),
              'ambiguity': ('ambiguity', 1), '-agree': ('agrees_with_student', -1),
              '-nbr_pl': ('nbr_pl_agree', -1), 'knn_pl_ent': ('knn_pl_entropy', 1)}
    rows = []
    for ds in datasets:
        run = PROBE / ds / 'standard/published'
        key = ds.split('_')[0]
        for seed in seeds:
            r = run / f'seed{seed}'
            if not (r / 'steps.jsonl').exists():
                continue
            steps = [json.loads(l) for l in (r / 'steps.jsonl').read_text().splitlines()]
            for direction in ('gnn->lm', 'lm->gnn'):
                sel = [s for s in steps if s['direction'] == direction]
                if not sel:
                    continue
                st = sel[0]
                F, wrong, z = features_for_step(r, st, key)
                va, te = z['valid_x'], z['test_x']
                if wrong[te].sum() < 5 or (~wrong[te]).sum() < 5:
                    continue
                out = {'dataset': ds, 'direction': direction, 'seed': seed,
                       'n_test': len(te), 'teacher_err_test': float(wrong[te].mean())}
                for name, (f, sgn) in SINGLE.items():
                    out[name] = fast_auroc(sgn * F[f][te], wrong[te])
                X = np.column_stack([F[k] for k in sorted(F)])
                sc = StandardScaler().fit(X[va])
                clf = LogisticRegression(max_iter=2000, C=1.0)
                clf.fit(sc.transform(X[va]), wrong[va])
                out['learned(val-fit)'] = fast_auroc(
                    clf.predict_proba(sc.transform(X[te]))[:, 1], wrong[te])
                rows.append(out)
    return pd.DataFrame(rows)


def gate_ablation(datasets, seed=0):
    """Does adding the pseudo-label features help the learned gate?

    Three fits per cell, all val-fit / test-evaluated:
      no_pl    everything except the two pseudo-label features
      with_pl  everything
      pl_only  the two pseudo-label features alone

    Also reports selection precision at the oracle's keep-rate -- of the N nodes
    the gate keeps, how many is the teacher actually right on. That is the quantity
    a gate acts on; AUROC only ranks.
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    PL_FEATS = {'nbr_pl_agree', 'knn_pl_entropy'}
    rows = []
    for ds in datasets:
        key = ds.split('_')[0]
        r = PROBE / ds / 'standard/published' / f'seed{seed}'
        if not (r / 'steps.jsonl').exists():
            continue
        steps = [json.loads(l) for l in (r / 'steps.jsonl').read_text().splitlines()]
        for direction in ('gnn->lm', 'lm->gnn'):
            sel = [s for s in steps if s['direction'] == direction]
            if not sel:
                continue
            F, wrong, z = features_for_step(r, sel[0], key)
            va, te = z['valid_x'], z['test_x']
            if wrong[te].sum() < 5 or (~wrong[te]).sum() < 5:
                continue
            out = {'dataset': key, 'direction': direction, 'n_test': len(te)}
            for tag, keys in [('no_pl', sorted(k for k in F if k not in PL_FEATS)),
                              ('with_pl', sorted(F)),
                              ('pl_only', sorted(PL_FEATS))]:
                X = np.column_stack([F[k] for k in keys])
                if not np.isfinite(X).all():      # knn_pl_entropy is NaN on big graphs
                    out[tag] = np.nan
                    continue
                sc = StandardScaler().fit(X[va])
                clf = LogisticRegression(max_iter=2000).fit(sc.transform(X[va]), wrong[va])
                score = clf.predict_proba(sc.transform(X[te]))[:, 1]
                out[tag] = fast_auroc(score, wrong[te])
                if tag == 'with_pl':
                    N = int((~wrong[te]).sum())    # the oracle's keep-count
                    out['prec_random'] = float((~wrong[te]).mean())
                    out['prec_gated'] = float((~wrong[te])[np.argsort(score)[:N]].mean())
            rows.append(out)
    d = pd.DataFrame(rows)
    d['delta_pl'] = (d.with_pl - d.no_pl).round(4)
    return d

In [2]:
pd.set_option('display.width', 240)
DATASETS = ['cora_TAG', 'pubmed_TAG', 'citeseer_TAG',
            'cornell_TAG+revgat', 'wisconsin_TAG+revgat', 'washington_TAG+revgat']

auroc = evaluate_gates(DATASETS)
print('AUROC for predicting TEACHER IS WRONG, on TEST nodes (fit on val where fitting is needed)')
print(auroc[['dataset', 'direction', 'n_test', 'teacher_err_test', '-conf', '-glance',
             'ambiguity', '-agree', '-nbr_pl', 'knn_pl_ent',
             'learned(val-fit)']].round(3).to_string(index=False))

abl = gate_ablation(DATASETS)
print()
print('LEARNED GATE ABLATION — delta_pl is the gain from adding the pseudo-label features')
print('prec_* = of the N nodes kept at the oracle keep-count, fraction the teacher is right on')
print(abl[['dataset', 'direction', 'n_test', 'no_pl', 'with_pl', 'delta_pl', 'pl_only',
           'prec_random', 'prec_gated']].round(3).to_string(index=False))

AUROC for predicting TEACHER IS WRONG, on TEST nodes (fit on val where fitting is needed)
              dataset direction  n_test  teacher_err_test  -conf  -glance  ambiguity  -agree  -nbr_pl  knn_pl_ent  learned(val-fit)
             cora_TAG   gnn->lm     542             0.116  0.870    0.868      0.746   0.571    0.809       0.783             0.891
             cora_TAG   lm->gnn     542             0.159  0.802    0.884      0.761   0.722    0.839       0.796             0.909
           pubmed_TAG   gnn->lm    3944             0.053  0.890    0.775      0.735   0.539    0.686       0.717             0.848
           pubmed_TAG   lm->gnn    3944             0.051  0.896    0.780      0.739   0.581    0.689       0.700             0.876
         citeseer_TAG   gnn->lm    2566             0.504  0.646    0.728      0.635   0.513    0.690       0.662             0.774
         citeseer_TAG   lm->gnn    2566             0.775  0.559    0.591      0.542   0.705    0.502       0.516     

### What the two tables say

**Single signals (first table).** Confidence and its relatives dominate on the large
datasets (0.87–0.90 on cora and pubmed). GLANCE wins where confidence is weak — cora
lm→gnn 0.884 vs 0.802, citeseer gnn→lm 0.728 vs 0.646. No single signal wins everywhere.

**The pseudo-label features (second table).** `delta_pl` is the change from adding both
to the learned gate: **+0.005, +0.010, +0.014 on three cells, −0.002, −0.005, −0.008,
−0.012 and −0.086 on five.** Four up, five down, and the biggest single movement is a
loss. They do not earn a place in the gate.

The reason is diagnostic rather than incidental: **`nbr_pl_agree` is inverted on the
heterophilous graphs** (AUROC 0.35–0.42 on WebKB — worse than chance). On a graph with
0.09–0.21 homophily, neighbours sharing your pseudo-label is evidence *against*
correctness. The feature carries real signal with a sign that flips by dataset, so a
globally-fit gate cannot use it.

The other half of the idea did pay off. **`knn_pl_entropy` beats the train-label
`ambiguity`** exactly where the train pool is thin — cora 0.783 vs 0.746, citeseer 0.662
vs 0.635, wisconsin gnn→lm 0.882 vs 0.809 — just not by enough to move the combination.

**The learned gate wins 6 of 10 cells** and is close elsewhere. Its two losses are
informative: on **pubmed** raw confidence beats it (0.890 vs 0.848), because with 3
classes confidence is nearly sufficient and the extra features only add variance; on
**washington** (n=22) it is noise.

**Precision is the number that matters**, since a gate thresholds rather than ranks. At
the oracle's own keep-count:

| dataset | direction | random | gated | oracle |
|---|---|---|---|---|
| cora | lm→gnn | 0.841 | **0.936** | 1.000 |
| citeseer | gnn→lm | 0.496 | **0.708** | 1.000 |
| citeseer | lm→gnn | 0.225 | **0.539** | 1.000 |
| pubmed | gnn→lm | 0.947 | **0.970** | 1.000 |
| wisconsin | gnn→lm | 0.692 | **0.889** | 1.000 |

The gate closes roughly half to two-thirds of the random→oracle gap, and most where the
teacher is weakest — citeseer lm→gnn goes from 23% correct to 54%.

**Conclusion.** Use a learned combination fit on val, over confidence / entropy / margin /
GLANCE / ambiguity / log-degree / teacher–student agreement, **excluding**
`nbr_pl_agree` because its sign flips with graph homophily. Keep `knn_pl_entropy` as an
optional swap for `ambiguity` where the train split is thin.

Two limits on this section: `knn_pl_entropy` is NaN on arxiv pending a GPU pass, and every
WebKB row rests on 19–26 test nodes, so cora, citeseer and pubmed are carrying the
analysis — the four small datasets are directional only.